<div id="singlestore-header" style="display: flex; background-color: #525252; padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/brand/database.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%; color: #FFFFFF;">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px; color: #FFFFFF;">Build an Application-Ready Marketing Transactions Schema with Sequences</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>This notebook requires a Standard or Premium cluster. Sequences are not available on Free Starter Clusters.</p>
    </div>
</div>

## Prerequisite: Generate the Real-Time Digital Marketing dataset

This notebook reads source data from the `martech` database populated by the RTDM demo. **Complete the steps below before running any code cells.**

### Two databases — know which is which

<table>
  <tr><th>Database</th><th>Owner</th><th>What this notebook does with it</th></tr>
  <tr><td><code>martech</code></td><td>RTDM demo</td><td><strong>Read-only.</strong> Never dropped or modified by this notebook.</td></tr>
  <tr><td><code>mktx_sequences_demo</code></td><td>This notebook</td><td>Created here. All DDL and application tables go here.</td></tr>
</table>

### Steps to populate martech

1. Make sure you are on a **Standard or Premium cluster** — the RTDM demo requires it.
2. Open the Real-Time Digital Marketing demo: **https://digital-marketing.labs.singlestore.com/**
3. Enter your cluster host, username `admin`, your cluster password, and database name `martech`.
4. Click **Connect**, then **Setup Database**.
5. Keep the simulator running until the dashboard shows purchases and offers being generated.
6. Return to this notebook.

### Before running DDL cells

Select **`mktx_sequences_demo`** from the database dropdown at the top of this notebook. The DDL creates tables in `mktx_sequences_demo` — not in `martech`.

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select <tt style="color: inherit; background-color: rgba(0,0,0,0.08);">mktx_sequences_demo</tt> from the database drop-down at the top of this notebook before running any table-creation cells.</p>
    </div>
</div>

## The application scenario

A subscriber just made a purchase through the RTDM platform. Your application needs to:

1. Record the purchase as a **marketing transaction** with a compact, unique identifier
2. Create a linked **order record** and derive a human-readable invoice display number
3. Attach one or more **line items** to that order
4. Do all of this safely under concurrent writes — without using `MAX(id) + 1`

By the end of this notebook your application tables will produce output like this:

<table>
  <tr><th>txn_id</th><th>display_ref</th><th>invoice_display</th><th>advertiser</th><th>vendor</th><th>bid_cents</th></tr>
  <tr><td>1</td><td>TXN-00001</td><td>INV-000001</td><td>Quatz</td><td>quatz.biz</td><td>10</td></tr>
  <tr><td>2</td><td>TXN-00002</td><td>INV-000002</td><td>Topiclounge</td><td>topiclounge.com</td><td>10</td></tr>
</table>

> Sequence values are database-generated — your actual `txn_id` and `order_id` values will differ.

**Reference:** [A Developer's Guide to SingleStore Sequences](https://www.singlestore.com/blog/guide-to-singlestore-sequences/)

## How the pieces fit together

```
RTDM purchase event  (subscriber_id, vendor_name, ts)
          │
          ▼
  marketing_txns      txn_id  ← sequence-generated
          │
          ▼
    mktx_orders       order_id ← sequence-generated
                      invoice_display = CONCAT('INV-', LPAD(order_id, 6, '0'))
          │
          ▼
  txn_line_items      line_id  ← sequence-generated
```

The source event columns travel with each transaction row for lineage. Sequence columns are generated by the database — the application never computes them.

## Section 1 — Environment setup

Installs dependencies, connects to SingleStore, and creates `mktx_sequences_demo`.

**Two SQL patterns used in this notebook:**
- **`%%sql`** — DDL, control statements, and display SELECTs.
- **`run_sql()`** — where Python is needed: parameterized queries, polling loops, and DataFrame results.

In [1]:
%pip install singlestoredb pandas --quiet

In [2]:
import random
import time
import pandas as pd
import singlestoredb as s2

pd.set_option("display.max_colwidth", 120)
random.seed(42)

# ── Database names ────────────────────────────────────────────────────────────
SOURCE_DB  = "martech"           # read-only — populated by the RTDM demo
WORKING_DB = "mktx_sequences_demo"  # this notebook's database — all DDL goes here
USE_DB     = None

# ── Connection helpers ────────────────────────────────────────────────────────
# run_sql is used where Python control flow, parameters, or DataFrame consumption
# is needed. Simple user-facing SQL cells use %%sql (see below).
def connect():
    global conn
    conn = s2.connect(database=USE_DB) if USE_DB else s2.connect()
    conn.autocommit(True)
    if USE_DB:
        with conn.cursor() as cur:
            cur.execute(f"USE {USE_DB}")

def _query(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description:
            cols = [d[0] for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)
    return None

def run_sql(query, params=None):
    for attempt in range(4):
        try:
            return _query(query, params)
        except Exception as e:
            msg = str(e)
            if getattr(e, "errno", None) in (2006, 2013) or "Lost connection" in msg or "gone away" in msg:
                connect()
            else:
                raise

def try_sql(query, params=None, label="Step"):
    try:
        return run_sql(query, params)
    except Exception as e:
        print(f"{label} skipped: {e}")
        return None

connect()

shared = run_sql("SHOW VARIABLES LIKE 'is_shared_tier'")
IS_SHARED = shared is not None and len(shared) > 0 and shared.iloc[0, 1] == "ON"

if IS_SHARED:
    print("Free Starter Cluster detected — using the current database.")
    USE_DB = run_sql("SELECT DATABASE() AS db").iloc[0, 0]
else:
    run_sql(f"CREATE DATABASE IF NOT EXISTS {WORKING_DB}")
    USE_DB = WORKING_DB
    run_sql(f"USE {WORKING_DB}")

connect()
active_db = run_sql("SELECT DATABASE() AS db").iloc[0, 0]
print(f"Source database  : {SOURCE_DB}")
print(f"Working database : {active_db}")
print(f"SingleStore version: {run_sql('SELECT @@memsql_version AS v').iloc[0, 0]}")

## Section 2 — Inspect the Real-Time Digital Marketing purchase data

Reads a bounded sample from `martech` and copies it into notebook-owned tables in `mktx_sequences_demo` (up to 10,000 purchases and 500 offers). Stops with a clear message if `martech` is empty.

### Create the RTDM source tables

Drops and recreates the three source tables scoped to this notebook. Safe to rerun.

In [3]:
%%sql
DROP TABLE IF EXISTS rtdm_purchases;
DROP TABLE IF EXISTS rtdm_offers;
DROP TABLE IF EXISTS rtdm_vendors;

CREATE TABLE rtdm_purchases (
    subscriber_id BIGINT,
    vendor_name   VARCHAR(100),
    ts            DATETIME(6) DEFAULT NOW(6),
    SORT KEY (ts),
    KEY (vendor_name) USING HASH
);

CREATE ROWSTORE TABLE rtdm_offers (
    offer_id             BIGINT PRIMARY KEY,
    customer             VARCHAR(100),
    enabled              BOOL,
    notification_zone    TEXT,
    segment_ids          JSON,
    notification_content TEXT,
    notification_target  VARCHAR(200),
    maximum_bid_cents    BIGINT
);

CREATE ROWSTORE REFERENCE TABLE rtdm_vendors (
    domain     VARCHAR(200) PRIMARY KEY,
    vendor     VARCHAR(100),
    category   VARCHAR(50),
    popularity DOUBLE
);

### Load RTDM data

Loads purchases, offers, and vendor reference data from `martech`. Because one vendor name can appear under multiple domains (e.g. `dabshots.mil`, `dabshots.edu`), a `rtdm_vendor_canonical` view picks exactly one domain per vendor name — preventing duplicate rows when we join purchases to vendors. This is a demo convenience, not a verified match between a purchase and an advertising campaign.

In [4]:
MAX_ROWS = 10_000

# ── Validate martech source tables ───────────────────────────────────────────
def _table_row_count(db, table):
    try:
        n = run_sql(
            "SELECT COUNT(*) AS n FROM information_schema.TABLES "
            "WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s", (db, table)
        ).iloc[0, 0]
        if int(n) == 0:
            return 0  # table does not exist
        return int(run_sql(f"SELECT COUNT(*) AS n FROM {db}.{table}").iloc[0, 0])
    except Exception:
        return 0

purchases_n = _table_row_count(SOURCE_DB, "purchases")
offers_n    = _table_row_count(SOURCE_DB, "offers")

if purchases_n == 0 or offers_n == 0:
    missing = []
    if purchases_n == 0:
        missing.append("purchases")
    if offers_n == 0:
        missing.append("offers")
    print("━" * 60)
    print(f"⛔ STOP: martech.{' and martech.'.join(missing)} {'is' if len(missing) == 1 else 'are'} empty or missing.")
    print()
    print("The martech database exists but has not been populated yet.")
    print("To fix this:")
    print("  1. Go to https://digital-marketing.labs.singlestore.com/")
    print("  2. Connect using your cluster host, username admin, and password.")
    print("  3. Set database name to: martech")
    print("  4. Click Setup Database.")
    print("  5. Keep the simulator running until purchases and offers appear.")
    print("  6. Return here and rerun this cell.")
    print("━" * 60)
    raise RuntimeError(f"martech source tables not ready: {missing}")

print(f"martech source validated — purchases: {purchases_n:,}, offers: {offers_n:,}")

# ── Vendors (always loaded from the RTDM GitHub repo) ────────────────────────
vdf = pd.read_json(
    "https://raw.githubusercontent.com/singlestore-labs/demo-realtime-digital-marketing/main/data/vendors.json"
)
vdf["domain"] = (vdf["vendor"].str.lower() + "." + vdf["tld"])
vdf = vdf.drop_duplicates("domain")
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO rtdm_vendors VALUES (%s, %s, %s, %s)",
        vdf[["domain", "vendor", "category", "popularity"]].values.tolist(),
    )
print(f"Vendors loaded: {len(vdf)} rows ({vdf['vendor'].nunique()} unique names)")

# One canonical domain per vendor name (demo enrichment, not confirmed attribution).
run_sql("DROP VIEW IF EXISTS rtdm_vendor_canonical")
run_sql("""
    CREATE VIEW rtdm_vendor_canonical AS
    SELECT v.vendor, v.domain, v.category
    FROM   rtdm_vendors v
    JOIN   (SELECT vendor, MIN(domain) AS domain FROM rtdm_vendors GROUP BY vendor) m
           ON m.vendor = v.vendor AND m.domain = v.domain
""")
print("Canonical vendor view created.")

# ── Load bounded sample from martech into mktx_sequences_demo ────────────────
run_sql(f"""
    INSERT INTO rtdm_purchases (subscriber_id, vendor_name, ts)
    SELECT subscriber_id, vendor, ts
    FROM {SOURCE_DB}.purchases
    ORDER BY ts DESC
    LIMIT {MAX_ROWS}
""")
run_sql(f"""
    INSERT IGNORE INTO rtdm_offers
    SELECT offer_id, customer, enabled, notification_zone,
           segment_ids, notification_content, notification_target, maximum_bid_cents
    FROM {SOURCE_DB}.offers
    ORDER BY offer_id
    LIMIT 500
""")
print("Loaded from martech.")

# ── Post-load verification ────────────────────────────────────────────────────
counts = run_sql('''
    SELECT
        (SELECT COUNT(*) FROM rtdm_purchases) AS rtdm_purchases,
        (SELECT COUNT(*) FROM rtdm_offers)    AS rtdm_offers,
        (SELECT COUNT(*) FROM rtdm_vendors)   AS rtdm_vendors
''')

p_loaded = int(counts["rtdm_purchases"].iloc[0])
o_loaded = int(counts["rtdm_offers"].iloc[0])

if p_loaded == 0 or o_loaded == 0:
    print("━" * 60)
    print("⛔ STOP: Working tables loaded zero rows.")
    print("  rtdm_purchases:", p_loaded)
    print("  rtdm_offers   :", o_loaded)
    print("Check that martech is populated and rerun from Section 2.")
    print("━" * 60)
    raise RuntimeError("Working tables are empty after load.")

display(counts)
print("✔ Source data loaded into mktx_sequences_demo.")

### Inspect a sample

Confirm the source columns the transaction layer will consume.

In [5]:
print("--- purchases (source event: decoded vendor name) ---")
display(run_sql(
    "SELECT subscriber_id, vendor_name, ts FROM rtdm_purchases ORDER BY ts DESC LIMIT 5"
))

print("\n--- canonical vendor enrichment (one domain per vendor name, demo only) ---")
display(run_sql(
    "SELECT vendor, domain, category FROM rtdm_vendor_canonical ORDER BY vendor LIMIT 5"
))

print("\n--- row-count check: purchases after canonical enrichment join ---")
display(run_sql('''
    SELECT
        COUNT(*)                                AS purchase_rows,
        COUNT(DISTINCT p.subscriber_id)         AS unique_subscribers,
        COUNT(DISTINCT vc.domain)               AS distinct_enriched_domains,
        COUNT(DISTINCT o.offer_id)              AS distinct_offers
    FROM      rtdm_purchases       p
    JOIN      rtdm_vendor_canonical vc ON vc.vendor = p.vendor_name
    JOIN      rtdm_offers           o  ON o.notification_target = vc.domain
'''))

print("\n--- offers sample ---")
display(run_sql(
    "SELECT offer_id, customer, notification_target AS domain, maximum_bid_cents "
    "FROM rtdm_offers ORDER BY offer_id LIMIT 5"
))

### Setup verification

Confirm both databases are visible and working tables contain data before proceeding.

In [6]:
print(f"Active working database : {run_sql('SELECT DATABASE() AS db').iloc[0, 0]}")
print(f"Source database         : {SOURCE_DB}")
print()

display(run_sql(f"""
    SELECT
        '{SOURCE_DB}'  AS source_db,
        (SELECT COUNT(*) FROM {SOURCE_DB}.purchases) AS source_purchases,
        (SELECT COUNT(*) FROM {SOURCE_DB}.offers)    AS source_offers
"""))

display(run_sql(f"""
    SELECT
        '{WORKING_DB}'  AS working_db,
        (SELECT COUNT(*) FROM rtdm_purchases) AS rtdm_purchases,
        (SELECT COUNT(*) FROM rtdm_offers)    AS rtdm_offers,
        (SELECT COUNT(*) FROM rtdm_vendors)   AS rtdm_vendors
"""))

rp = int(run_sql("SELECT COUNT(*) AS n FROM rtdm_purchases").iloc[0, 0])
ro = int(run_sql("SELECT COUNT(*) AS n FROM rtdm_offers").iloc[0, 0])

if rp > 0 and ro > 0:
    print("✔ Ready — working tables contain data.")
else:
    print("⚠ Working tables are empty. Rerun from Section 2 after populating martech.")

## Section 3 — Why sequences instead of `MAX(id) + 1`

A common pattern is to compute the next ID in application code:

```python
next_id = SELECT MAX(txn_id) + 1 FROM marketing_txns
INSERT INTO marketing_txns (txn_id, ...) VALUES (next_id, ...)
```

This breaks under concurrent writes: two sessions can read the same `MAX` simultaneously and both try to insert the same ID, producing a **duplicate key error**. `AUTO_INCREMENT AS SEQUENCE` lets the database generate IDs — each aggregator pre-caches a block of values, so every insert gets a unique ID with no coordination required.

## Section 4 — Create application-owned tables with sequences

Three tables form the application boundary, each using `INT AUTO_INCREMENT AS SEQUENCE` as its primary key.

<table>
  <tr><th>Table</th><th>Purpose</th><th>Sequence column</th></tr>
  <tr><td><code>marketing_txns</code></td><td>One row per accepted purchase event</td><td><code>txn_id</code></td></tr>
  <tr><td><code>mktx_orders</code></td><td>One order per transaction; source of the invoice display number</td><td><code>order_id</code></td></tr>
  <tr><td><code>txn_line_items</code></td><td>One row per vendor line within a transaction</td><td><code>line_id</code></td></tr>
</table>

> A sequence PK is incompatible with a separate composite unique key in SingleStore. Idempotency is handled at the application layer with `TRUNCATE` before each full load.

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Make sure <tt style="color: inherit; background-color: rgba(0,0,0,0.08);">mktx_sequences_demo</tt> is selected from the database drop-down at the top of this notebook before running the cells below. The DDL creates tables in <tt style="color: inherit; background-color: rgba(0,0,0,0.08);">mktx_sequences_demo</tt> — not in <tt style="color: inherit; background-color: rgba(0,0,0,0.08);">martech</tt>. The <tt style="color: inherit; background-color: rgba(0,0,0,0.08);">martech</tt> source database is never dropped or modified by this notebook.</p>
    </div>
</div>

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-info-circle"></b>
    <div>
        <p><b>One sequence per table</b></p>
        <p>SingleStore supports exactly <strong>one</strong> <code style="color: inherit; background-color: rgba(0,0,0,0.08);">AUTO_INCREMENT AS SEQUENCE</code> column per table. Do not add a second sequence column to the same table.</p>
    </div>
</div>

In [7]:
%%sql
DROP TABLE IF EXISTS txn_line_items;
DROP TABLE IF EXISTS mktx_orders;
DROP TABLE IF EXISTS marketing_txns;

-- Note: with a sequence PK, a separate composite unique key is not supported.
-- Idempotency is handled at the application layer (see Section 5).
CREATE ROWSTORE TABLE marketing_txns (
    txn_id          INT  AUTO_INCREMENT AS SEQUENCE PRIMARY KEY,
    subscriber_id   BIGINT       NOT NULL,
    event_ts        DATETIME(6)  NOT NULL,
    offer_id        BIGINT       NOT NULL,
    customer        VARCHAR(100),
    domain          VARCHAR(200),
    category        VARCHAR(50),
    bid_cents       BIGINT,
    created_at      DATETIME(6) DEFAULT NOW(6),
    KEY idx_src (subscriber_id, event_ts, offer_id) USING HASH
);

CREATE ROWSTORE TABLE mktx_orders (
    order_id    INT  AUTO_INCREMENT AS SEQUENCE PRIMARY KEY,
    txn_id      INT  NOT NULL,
    customer    VARCHAR(100),
    domain      VARCHAR(200),
    bid_cents   BIGINT,
    created_at  DATETIME(6) DEFAULT NOW(6),
    KEY idx_txn (txn_id) USING HASH
);

CREATE ROWSTORE TABLE txn_line_items (
    line_id     INT  AUTO_INCREMENT AS SEQUENCE PRIMARY KEY,
    txn_id      INT  NOT NULL,
    vendor      VARCHAR(100),
    category    VARCHAR(50),
    bid_cents   BIGINT,
    KEY idx_txn (txn_id) USING HASH
);

### Verify the sequence columns

`SHOW CREATE TABLE` confirms that `txn_id` uses `AUTO_INCREMENT AS SEQUENCE`.

In [8]:
%%sql
-- Confirm the active database and that txn_id uses AUTO_INCREMENT AS SEQUENCE
SELECT DATABASE();
SHOW CREATE TABLE marketing_txns;

## Section 5 — Sequence demonstration: insert transactions and link child records

This section shows sequences in action across three tables:

1. **Bulk insert** — 200 purchase events are inserted into `marketing_txns`. No `txn_id` is supplied; the sequence generates one for each row automatically.
2. **Linked orders** — one `mktx_orders` row is created per transaction, copying the sequence-generated `txn_id` as a foreign key.
3. **Parent-child linking** — a single event is inserted step-by-step to show how `LAST_INSERT_ID()` captures the generated `txn_id` and passes it to the child `txn_line_items` row.

The tables are truncated first so this section is safe to rerun.

In [9]:
%%sql
TRUNCATE TABLE txn_line_items;
TRUNCATE TABLE mktx_orders;
TRUNCATE TABLE marketing_txns;

In [10]:
MAX_ROWS = 200

# Parameterized — uses Python variable for LIMIT, so run_sql() is needed here.
run_sql(f"""
    INSERT INTO marketing_txns
        (subscriber_id, event_ts, offer_id, customer, domain, category, bid_cents)
    SELECT
        p.subscriber_id,
        p.ts,
        o.offer_id,
        o.customer,
        vc.domain,
        COALESCE(vc.category, 'general'),
        o.maximum_bid_cents
    FROM       rtdm_purchases        p
    JOIN       rtdm_vendor_canonical vc ON vc.vendor             = p.vendor_name
    JOIN       rtdm_offers           o  ON o.notification_target = vc.domain
    ORDER BY   p.ts DESC
    LIMIT {MAX_ROWS}
""")
run_sql("""
    INSERT INTO mktx_orders (txn_id, customer, domain, bid_cents)
    SELECT txn_id, customer, domain, bid_cents
    FROM   marketing_txns
""")
print(f"Inserted up to {MAX_ROWS} transactions and linked orders.")

In [11]:
%%sql
SELECT
    (SELECT COUNT(*) FROM marketing_txns) AS txns_loaded,
    (SELECT COUNT(*) FROM mktx_orders)   AS orders_created;

### Inspect the generated identifiers

The source event columns travel alongside the database-generated `txn_id` and serve different purposes:

<table>
  <tr><th>Column</th><th>Purpose</th></tr>
  <tr><td><code>txn_id</code></td><td>Compact application identifier — generated by the sequence, never supplied by the caller</td></tr>
  <tr><td><code>subscriber_id</code>, <code>event_ts</code>, <code>offer_id</code></td><td>Source identity — used for lineage and duplicate detection</td></tr>
</table>

In [12]:
%%sql
SELECT txn_id, subscriber_id, event_ts,
       offer_id, customer, category, bid_cents
FROM   marketing_txns
ORDER  BY txn_id
LIMIT  10;

### Using `LAST_INSERT_ID()` to link a parent row to its children

After the database generates a `txn_id` for an inserted transaction, `LAST_INSERT_ID()` returns that exact value for the current session. This is how you safely pass a sequence-generated ID from a parent row to its child rows — without doing a follow-up `SELECT MAX(txn_id)` that could return a different session's row under concurrent writes.

In [13]:
# Insert one conversion event and link its order and line items via LAST_INSERT_ID()
sample = run_sql('''
    SELECT p.subscriber_id,
           p.ts          AS event_ts,
           o.offer_id, o.customer,
           vc.domain,
           COALESCE(vc.category, 'general') AS category,
           o.maximum_bid_cents
    FROM      rtdm_purchases        p
    JOIN      rtdm_vendor_canonical vc ON vc.vendor             = p.vendor_name
    JOIN      rtdm_offers           o  ON o.notification_target = vc.domain
    ORDER BY  p.ts
    LIMIT 1
''')

row = sample.iloc[0]

with conn.cursor() as cur:
    # Insert the parent transaction — no txn_id supplied
    cur.execute('''
        INSERT INTO marketing_txns
            (subscriber_id, event_ts, offer_id, customer, domain, category, bid_cents)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    ''', (int(row.subscriber_id), row.event_ts, int(row.offer_id),
           row.customer, row.domain, row.category, int(row.maximum_bid_cents)))

    # Retrieve the sequence value generated for this session
    cur.execute('SELECT LAST_INSERT_ID() INTO @parent_txn_id')

    cur.execute('''
        INSERT INTO mktx_orders (txn_id, customer, domain, bid_cents)
        VALUES (@parent_txn_id, %s, %s, %s)
    ''', (row.customer, row.domain, int(row.maximum_bid_cents)))

    cur.execute('''
        INSERT INTO txn_line_items (txn_id, vendor, category, bid_cents)
        VALUES (@parent_txn_id, %s, %s, %s)
    ''', (row.domain, row.category, int(row.maximum_bid_cents)))

run_sql('''
    SELECT t.txn_id,
           CONCAT('INV-', LPAD(ord.order_id, 6, '0')) AS invoice_ref,
           t.subscriber_id, t.customer, t.category,
           l.line_id, l.vendor, l.bid_cents
    FROM   marketing_txns  t
    JOIN   mktx_orders     ord ON ord.txn_id = t.txn_id
    JOIN   txn_line_items  l   ON l.txn_id   = t.txn_id
    ORDER  BY t.txn_id DESC
    LIMIT  5
''')

### Preventing duplicate transactions

Because `txn_id` is sequence-generated, you cannot add a composite unique constraint to block re-inserts at the database level. Instead, check whether a transaction already exists for the source event key `(subscriber_id, event_ts, offer_id)` before inserting — and skip if one is found. The next cell demonstrates this guard.

In [14]:
# Attempt to insert the same purchase event a second time.
# The guard checks whether a transaction already exists for this source event
# before inserting, so no duplicate application record is created.
sample = run_sql('''
    SELECT subscriber_id, event_ts, offer_id
    FROM   marketing_txns
    ORDER  BY txn_id
    LIMIT  1
''')
row = sample.iloc[0]

existing = run_sql('''
    SELECT txn_id FROM marketing_txns
    WHERE subscriber_id = %s AND event_ts = %s AND offer_id = %s
''', (int(row.subscriber_id), row.event_ts, int(row.offer_id)))

if len(existing) > 0:
    print(f"Source event already recorded as txn_id={existing.iloc[0,0]} — skipping insert.")
    print("The source event key (subscriber_id, event_ts, offer_id) is the idempotency key.")
    print("The sequence-generated txn_id is the application key — these are separate concerns.")
else:
    run_sql('''
        INSERT INTO marketing_txns
            (subscriber_id, event_ts, offer_id, customer, domain, category, bid_cents)
        VALUES (%s, %s, %s, '', '', '', 0)
    ''', (int(row.subscriber_id), row.event_ts, int(row.offer_id)))
    print("Inserted (event was new).")

### Validate the generated identifiers

Confirm every sequence-generated key is unique and no line items are orphaned.

In [15]:
checks = {}

# Primary key uniqueness
dup_pk = run_sql('''
    SELECT txn_id, COUNT(*) AS cnt
    FROM   marketing_txns
    GROUP  BY txn_id
    HAVING cnt > 1
''')
checks["duplicate txn_id"] = len(dup_pk)

# Order uniqueness (one order per transaction)
dup_ord = run_sql('''
    SELECT order_id, COUNT(*) AS cnt
    FROM   mktx_orders
    GROUP  BY order_id
    HAVING cnt > 1
''')
checks["duplicate order_id"] = len(dup_ord)

# Orphaned line items (no parent txn)
orphans = run_sql('''
    SELECT COUNT(*) AS n
    FROM   txn_line_items l
    LEFT JOIN marketing_txns t ON t.txn_id = l.txn_id
    WHERE  t.txn_id IS NULL
''')
checks["orphaned line_items"] = int(orphans.iloc[0, 0])

for k, v in checks.items():
    status = "PASS" if v == 0 else "FAIL"
    print(f"[{status}] {k}: {v}")

## Section 6 — Sequence properties and the invoice display number

Key properties of SingleStore sequences:

<table>
  <tr><th>Property</th><th>Behaviour</th></tr>
  <tr><td><strong>Unique</strong></td><td>Every generated value is guaranteed unique</td></tr>
  <tr><td><strong>Not gapless</strong></td><td>Rolled-back transactions leave unused values; gaps are normal</td></tr>
  <tr><td><strong>Cached per aggregator</strong></td><td>Each aggregator pre-fetches a block of values; jumps in the sequence are expected</td></tr>
  <tr><td><strong>One per table</strong></td><td>Only one <code>AUTO_INCREMENT AS SEQUENCE</code> column is allowed per table</td></tr>
  <tr><td><strong>Not an idempotency key</strong></td><td>The sequence does not block duplicates; use the source event key for that</td></tr>
</table>

`CONCAT('INV-', LPAD(order_id, 6, '0'))` builds the invoice display number on the fly from `order_id`. It is a presentation label — not a stored column and not a gapless accounting sequence.

In [16]:
%%sql
-- Invoice display number derived in SQL — not stored as a column
SELECT t.txn_id,
       CONCAT("TXN-", LPAD(t.txn_id, 5, "0"))        AS display_ref,
       ord.order_id,
       CONCAT("INV-", LPAD(ord.order_id, 6, "0"))     AS invoice_display_number,
       t.customer, t.category, t.bid_cents
FROM   marketing_txns t
JOIN   mktx_orders    ord ON ord.txn_id = t.txn_id
ORDER  BY t.txn_id
LIMIT  8;

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Manual override warning</b></p>
        <p>Manually supplying a value for a sequence column (e.g. <code style="color: inherit; background-color: rgba(0,0,0,0.08);">INSERT INTO marketing_txns (txn_id, ...) VALUES (99, ...)</code>) can desynchronise the aggregator caches. If you ever do this, run:</p>
        <pre style="color: inherit; background-color: rgba(0,0,0,0.08); border: none;">AGGREGATOR SYNC AUTO_INCREMENT ON marketing_txns ALL;</pre>
        <p>This notebook does <strong>not</strong> perform manual overrides in the main path.</p>
    </div>
</div>

## Section 7 — API and support lookup

The query an API endpoint, support workflow, or AI assistant would run to retrieve full transaction context: purchase event, subscriber info, vendor enrichment, generated identifiers, and line items.

In [17]:
%%sql
SELECT
    t.txn_id,
    CONCAT("TXN-", LPAD(t.txn_id, 5, "0"))         AS display_ref,
    ord.order_id,
    CONCAT("INV-", LPAD(ord.order_id, 6, "0"))      AS invoice_display_number,
    t.subscriber_id,
    t.event_ts,
    t.customer                                        AS advertiser,
    t.category,
    t.domain                                          AS vendor_domain,
    t.bid_cents,
    l.line_id,
    l.vendor,
    l.bid_cents                                       AS line_bid_cents
FROM       marketing_txns  t
JOIN       mktx_orders     ord ON ord.txn_id = t.txn_id
LEFT JOIN  txn_line_items  l   ON l.txn_id   = t.txn_id
ORDER BY   t.txn_id DESC
LIMIT 5;

### Extending the lookup

Replace the `LIMIT 5` clause with `WHERE txn_id = N` to look up a single transaction. All context is stored in the application tables — no external credentials required.

In [18]:
%%sql
-- Runnable lookup by transaction ID — replace LIMIT with a WHERE txn_id = N clause to look up a single transaction.
SELECT
    t.txn_id,
    CONCAT("INV-", LPAD(ord.order_id, 6, "0"))  AS invoice_display_number,
    t.subscriber_id,
    t.event_ts,
    t.customer                                    AS advertiser,
    t.category,
    t.domain                                      AS vendor_domain,
    t.bid_cents,
    l.line_id,
    l.vendor,
    l.bid_cents                                   AS line_bid_cents
FROM       marketing_txns  t
JOIN       mktx_orders     ord ON ord.txn_id  = t.txn_id
LEFT JOIN  txn_line_items  l   ON l.txn_id    = t.txn_id
ORDER BY   t.txn_id DESC
LIMIT 5;

## Key takeaways

<table>
  <tr><th>Concept</th><th>What this notebook showed</th></tr>
  <tr><td><code>AUTO_INCREMENT AS SEQUENCE</code></td><td>Database-generated identifiers — unique, sequential, no application coordination needed</td></tr>
  <tr><td>Source event identity</td><td><code>(subscriber_id, event_ts, offer_id)</code> travels with each row for lineage and duplicate detection</td></tr>
  <tr><td>Parent-child linking</td><td><code>LAST_INSERT_ID()</code> returns the sequence value generated for the current session — safe under concurrent writes</td></tr>
  <tr><td>Sequence gaps</td><td>Rolled-back transactions leave gaps; non-contiguous values are normal and expected</td></tr>
  <tr><td>Invoice display number</td><td><code>CONCAT('INV-', LPAD(order_id, 6, '0'))</code> is a presentation-layer label, not an accounting invoice sequence</td></tr>
  <tr><td>Idempotency</td><td>With a sequence PK, composite unique constraints are not supported; guard duplicate events at the application layer</td></tr>
</table>

## Section 8 — Cleanup and rerun guidance

Run this cell to drop all objects created by this notebook, then re-execute from Section 2 for a clean start. `martech` is never modified.

In [19]:
app_tables = ["txn_line_items", "mktx_orders", "marketing_txns"]
src_tables = ["rtdm_purchases", "rtdm_offers", "rtdm_vendors"]

if not IS_SHARED:
    run_sql(f"DROP DATABASE IF EXISTS {WORKING_DB}")
    print(f"Database '{WORKING_DB}' dropped.")
else:
    for t in app_tables + src_tables:
        run_sql(f"DROP TABLE IF EXISTS {t}")
        print(f"Dropped table: {t}")

print("Cleanup complete. martech was not modified.")

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>